# 05 · Monte Carlo Methods
**TABULAR METHODS** · ⏱ 45 min · 🧪 3 exercises · ⭐ 400 XP

*Learn values and policies from complete episodes of experience — no model required. Blackjack, ε-soft control and importance sampling.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "05-monte-carlo"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

Dynamic programming was a planner's dream: hand it `p(s', r | s, a)` and it computes v_π, then
v*, exactly. But almost nothing interesting comes with that model attached. Nobody gives you the
transition probabilities of a Blackjack shoe, a robot's gearbox or a stock market.

**Monte Carlo (MC)** methods throw the model away. They learn from **sample episodes** — actual
sequences of states, actions and rewards — and they do the simplest thing imaginable with them:

> the value of a state is the average return you have seen after visiting it.

That is it. No Bellman backup, no `p(s'|s,a)`. Just play, write down what happened, average.

## 01 · Why averaging works

Recall the definition from lesson 3:

```text
v_π(s) = E_π[ Gₜ | Sₜ = s ]
```

An expectation is the long-run mean of samples. Every time an episode passes through *s*, the
return Gₜ that followed is one unbiased sample of v_π(s). By the law of large numbers the
average converges to v_π(s), and its standard error shrinks like 1/√n.

Three properties make MC very different from DP:

- **Model-free.** It needs experience, not dynamics.
- **No bootstrapping.** Each estimate is built from *real* returns, never from other estimates.
  If one state's value is wrong, its neighbours are not infected.
- **Episode-by-episode.** You only know Gₜ once the episode is *over*, so MC only works for
  episodic tasks and only learns at the end of each one.

> **ℹ️ NOTE** · **Cost independent of the state space**
>
> The work to estimate one state's value does not depend on how many other states there are.
> If you only care about the value of the starting position, you can simulate from there and
> ignore the rest of the world. DP has no such shortcut.

## 02 · Monte Carlo prediction

Given a policy π, play episodes, and for each episode walk **backwards** accumulating the
return with the recursion you wrote in lesson 1:

```text
G ← Rₜ₊₁ + γ G        (for t = T−1, T−2, …, 0)
```

and add G to the list of returns observed for Sₜ.

### First visit or every visit?

A state can appear several times in one episode. Two flavours handle that differently:

| | which returns count | properties |
|---|---|---|
| **first-visit MC** | only the return after the *first* occurrence of s in each episode | each sample is i.i.d.; unbiased; the textbook's default |
| **every-visit MC** | the return after *every* occurrence | samples within an episode are correlated; biased for finite n but still consistent |

A worked example. One episode, γ = 1:

| t | Sₜ | Rₜ₊₁ | Gₜ |
|---|---|---|---|
| 0 | A | 1 | 1 + 0 + 2 = **3** |
| 1 | A | 0 | 0 + 2 = **2** |
| 2 | B | 2 | **2** |

First-visit gives V(A) = 3. Every-visit gives V(A) = (3 + 2)/2 = 2.5. Both give V(B) = 2. With
many episodes both converge to v_π(A); they just get there differently.

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("05-monte-carlo#0", {"q": "Same episode (A, A, B with rewards 1, 0, 2) but γ = 0.5. What is the FIRST-visit estimate of V(A)?", "options": ["3", "1.5", "1.25", "2"], "answer": 1, "explain": "Walk backwards: G₂ = 2, G₁ = 0 + 0.5·2 = 1, G₀ = 1 + 0.5·1 = 1.5. First-visit uses G₀ = 1.5 (every-visit would average 1.5 and 1 → 1.25)."})

### The Blackjack testbed

Our main example is the classic from Sutton & Barto (Example 5.1), bundled as `rlenvs.Blackjack`.
You play against a dealer from an infinite deck. Face cards count 10, an ace counts 1 or 11.

- **state** `(player_sum, dealer_showing, usable_ace)` — e.g. `(15, 10, False)`
- **actions** `0 = stick`, `1 = hit`
- **reward** +1 win, 0 draw, −1 loss, given only at the end; γ = 1

An ace is *usable* if you can count it as 11 without busting — it is a free safety net, which is
why it is part of the state. Writing down the transition probabilities of this game is a
nightmare (what is p(next state | sum 14, hit) when the dealer then plays out its hand?).
Simulating it is ten lines of code. That gap is exactly where Monte Carlo shines.

Consider the naive policy *stick on 20 or 21, otherwise hit*. How good is it? Let Monte Carlo
tell you.

---
### 🧪 Exercise 1 · First-visit Monte Carlo prediction · ⭐ **120 XP**

Estimate v_π purely from sampled episodes — no model, no bootstrapping.

1. `generate_episode(env, policy, seed=None)` plays one episode and returns a list of `(state, action, reward)` triples `[(S₀, A₀, R₁), (S₁, A₁, R₂), …]`. Pass `seed` to `env.reset`, stop on `terminated or truncated`.
2. `mc_prediction(env, policy, n_episodes, gamma=1.0, first_visit=True, seed=None)` returns a dict `{state: V(state)}`: the average of the returns that followed visits to each state. With `first_visit=True` only the **first** visit of a state in each episode contributes; with `False`, every visit does. Use `seed` for the **first** episode's reset only (afterwards the env's own RNG keeps dealing new cards).

Walk each episode **backwards** with `G ← R + γ·G` — the same trick as `returns_to_go` in lesson 1.

The check first uses a tiny scripted env where state A is visited twice (so first- and every-visit give different answers you can work out by hand), then evaluates the *stick on 20* Blackjack policy. **Run** draws the two value surfaces of Sutton & Barto's Figure 5.1.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 05-monte-carlo/01-mc-prediction
from collections import defaultdict

import rlhud
from rlenvs import Blackjack


def generate_episode(env, policy, seed=None):
    """Play one episode with `policy(obs) -> action`.

    Returns a list of (state, action, reward) triples, where `reward` is the
    reward received *after* taking `action` in `state`:  [(S0, A0, R1), (S1, A1, R2), ...]
    """
    # TODO: reset (passing seed), then loop: pick an action, step, record (obs, action, reward)
    #       until terminated or truncated.
    raise NotImplementedError


def mc_prediction(env, policy, n_episodes, gamma=1.0, first_visit=True, seed=None):
    """Monte Carlo estimate of v_pi. Returns a dict {state: estimated value}.

    first_visit=True  -> only the first occurrence of a state in an episode counts
    first_visit=False -> every occurrence counts (every-visit MC)
    `seed` is passed to env.reset for the first episode only.
    """
    # TODO:
    #   for each episode (seed only on the first reset):
    #     1. generate it
    #     2. walk it BACKWARDS keeping G = r + gamma * G
    #     3. add G to returns_sum[s] / returns_count[s] — for first-visit, only when t is the
    #        first time s appears in the episode
    #   return {s: average return}
    raise NotImplementedError


def stick_on_20(obs):
    """The classic 'naive' Blackjack policy: hit until you reach 20 or 21."""
    player_sum, dealer_card, usable_ace = obs
    return 0 if player_sum >= 20 else 1


if __name__ == "__main__":
    env = Blackjack(seed=0)
    N = 50_000
    V = mc_prediction(env, stick_on_20, N, seed=0)

    # Lay V out like Sutton & Barto Figure 5.1: rows = player sum 21..12, cols = dealer A..10
    sums = list(range(21, 11, -1))
    dealer = list(range(1, 11))
    for usable in (False, True):
        grid = [[V.get((p, d, usable), 0.0) for d in dealer] for p in sums]
        labels = [[f"{V.get((p, d, usable), 0.0):.2f}" for d in dealer] for p in sums]
        rlhud.grid(grid, title=f"v_π, stick on 20 — {'usable' if usable else 'no usable'} ace "
                               f"(rows 21→12, cols dealer A→10)", labels=labels)
    rlhud.stat("Episodes", N)
    rlhud.stat("States seen", len(V))
    rlhud.stat("V(20, 10, no ace)", round(V.get((20, 10, False), float("nan")), 3))
    rlhud.stat("V(15, 10, no ace)", round(V.get((15, 10, False), float("nan")), 3))
    print("Sticking on 20 is great if you get there; every sum below 20 is a losing position.")

In [ ]:
#@title ✅ **Check** · First-visit Monte Carlo prediction (+120 XP) — tests the code you last ran above
rlcolab.check("05-monte-carlo/01-mc-prediction")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Keep two dicts, `returns_sum` and `returns_count` (a `collections.defaultdict` is handy). The estimate is their ratio — you could also keep a running mean.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Before the backward pass, record the first index at which each state appears: `first = {}; for t, (s, _, _) in enumerate(episode): first.setdefault(s, t)`. In the backward pass, skip time t when `first_visit and first[s] != t`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

G = 0.0
for t in range(len(episode) - 1, -1, -1):
    s, _, r = episode[t]
    G = r + gamma * G
    if first_visit and first[s] != t: continue
    returns_sum[s] += G; returns_count[s] += 1

…and finally `return {s: returns_sum[s] / returns_count[s] for s in returns_count}`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("05-monte-carlo/01-mc-prediction")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 05-monte-carlo/01-mc-prediction
# Reference solution
from collections import defaultdict

import rlhud
from rlenvs import Blackjack


def generate_episode(env, policy, seed=None):
    """Play one episode with `policy(obs) -> action`.

    Returns a list of (state, action, reward) triples, where `reward` is the
    reward received *after* taking `action` in `state`:  [(S0, A0, R1), (S1, A1, R2), ...]
    """
    episode = []
    obs, _ = env.reset(seed=seed)
    while True:
        action = policy(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        episode.append((obs, action, reward))
        if terminated or truncated:
            return episode
        obs = next_obs


def mc_prediction(env, policy, n_episodes, gamma=1.0, first_visit=True, seed=None):
    """Monte Carlo estimate of v_pi. Returns a dict {state: estimated value}.

    first_visit=True  -> only the first occurrence of a state in an episode counts
    first_visit=False -> every occurrence counts (every-visit MC)
    `seed` is passed to env.reset for the first episode only.
    """
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)
    for i in range(n_episodes):
        episode = generate_episode(env, policy, seed=seed if i == 0 else None)
        # Index of the first time each state appears (for first-visit MC).
        first = {}
        for t, (s, _, _) in enumerate(episode):
            first.setdefault(s, t)
        G = 0.0
        # Walk backwards: G_t = R_{t+1} + gamma * G_{t+1}
        for t in range(len(episode) - 1, -1, -1):
            s, _, r = episode[t]
            G = r + gamma * G
            if first_visit and first[s] != t:
                continue
            returns_sum[s] += G
            returns_count[s] += 1
    return {s: returns_sum[s] / returns_count[s] for s in returns_count}


def stick_on_20(obs):
    """The classic 'naive' Blackjack policy: hit until you reach 20 or 21."""
    player_sum, dealer_card, usable_ace = obs
    return 0 if player_sum >= 20 else 1


if __name__ == "__main__":
    env = Blackjack(seed=0)
    N = 50_000
    V = mc_prediction(env, stick_on_20, N, seed=0)

    # Lay V out like Sutton & Barto Figure 5.1: rows = player sum 21..12, cols = dealer A..10
    sums = list(range(21, 11, -1))
    dealer = list(range(1, 11))
    for usable in (False, True):
        grid = [[V.get((p, d, usable), 0.0) for d in dealer] for p in sums]
        labels = [[f"{V.get((p, d, usable), 0.0):.2f}" for d in dealer] for p in sums]
        rlhud.grid(grid, title=f"v_π, stick on 20 — {'usable' if usable else 'no usable'} ace "
                               f"(rows 21→12, cols dealer A→10)", labels=labels)
    rlhud.stat("Episodes", N)
    rlhud.stat("States seen", len(V))
    rlhud.stat("V(20, 10, no ace)", round(V.get((20, 10, False), float("nan")), 3))
    rlhud.stat("V(15, 10, no ace)", round(V.get((15, 10, False), float("nan")), 3))
    print("Sticking on 20 is great if you get there; every sum below 20 is a losing position.")

> **💡 TIP** · **Read the surface**
>
> After **Run**, look at the two grids. The top rows (20 and 21) are bright: sticking there wins
> most of the time. Everything below 20 is negative — this policy keeps hitting until it busts.
> The usable-ace surface is noisier: those states are rarer, so they have fewer samples.

## 03 · From prediction to control

To *improve* a policy without a model we need **action values**. With v_π alone, choosing the
best action requires a one-step look-ahead through `p(s'|s,a)` — the model we do not have. With
q_π you simply take `argmax_a Q(s, a)`. So model-free control estimates **Q(s, a)**: the average
return after taking *a* in *s* and following π afterwards.

Then we run **generalised policy iteration** (GPI) exactly as in lesson 4, with sampled episodes
doing the evaluation:

```text
π₀ → q_π₀ → π₁ → q_π₁ → π₂ → …       (evaluate by averaging, improve by acting greedily)
```

We do not wait for evaluation to converge — after every episode we update Q and the next episode
already uses the improved policy. GPI is happy with partial evaluation.

### The exploration problem

There is a catch. If π is deterministic and greedy, then from each state we only ever try one
action, so we only ever get returns for that one action. Q for the others stays at its initial
value forever and we can never discover they are better. Two classic fixes:

1. **Exploring starts.** Begin every episode in a random state with a random first action, so
   every (s, a) pair has a chance to be the start. It works in simulation, but you cannot tell a
   real robot to "start in a random state".
2. **ε-soft policies.** Never become fully greedy. Act **ε-greedily**: with probability ε pick a
   uniformly random action, otherwise the greedy one. Every action keeps probability ≥ ε/|A|.

```text
π(a|s) = 1 − ε + ε/|A|   if a = argmax Q(s, ·),   otherwise ε/|A|
```

The policy improvement theorem still holds *within the class of ε-soft policies*: ε-greedy with
respect to q_π is at least as good as π. So on-policy MC control converges to the best ε-soft
policy — not quite optimal, but close when ε is small.

In [ ]:
#@title 🎮 **Interactive** · ε-greedy distribution
rlcolab.widget("epsilon", values=[0.2, -0.4, 0.6, 0.1], epsilon=0.1)

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("05-monte-carlo#1", {"q": "Why does model-free control estimate Q(s, a) rather than V(s)?", "options": ["Q is smaller to store", "Choosing a greedy action from V needs the transition model; from Q it is just argmax", "V cannot be estimated by Monte Carlo"], "answer": 1, "explain": "argmaxₐ Σ p(s',r|s,a)[r + γV(s')] needs p. argmaxₐ Q(s,a) does not — the model is already baked into Q."})

### Incremental averages

Storing every return and re-averaging is wasteful. The incremental mean from the bandits lesson
does the same in O(1):

```text
N(s,a) ← N(s,a) + 1        Q(s,a) ← Q(s,a) + (G − Q(s,a)) / N(s,a)
```

Replace 1/N with a constant step α and you get a recency-weighted average that tracks a changing
target — useful because, in control, the policy (and hence the target) keeps changing.

---
### 🧪 Exercise 2 · On-policy MC control on Blackjack · ⭐ **150 XP**

Learn to play Blackjack from scratch with **on-policy first-visit Monte Carlo control**.

1. `epsilon_greedy(q, epsilon, rng)` — `q` is the array of action values for one state; with probability ε return a uniformly random action (`rng.integers(len(q))`), otherwise `argmax(q)`. `rng` is a `np.random.default_rng`.
2. `mc_control(env, n_episodes, epsilon=0.1, gamma=1.0, seed=None)` returns `(Q, returns)`:
   - `Q` is a dict `{state: np.zeros(env.n_actions)}`, created lazily the first time a state is seen.
   - each episode: act ε-greedily w.r.t. the current Q, record `(s, a, r)`; then walk it backwards with `G ← r + γG` and, for the **first** visit of each `(s, a)`, move Q(s,a) to the running mean of its returns: `N += 1; Q += (G − Q)/N`.
   - `returns` is the list of total rewards per episode.
   - make `rng = np.random.default_rng(seed)` and pass `seed` to the first `env.reset` only.

`evaluate(env, policy, n_episodes, seed)` is provided. The check plays 20 000 hands with your learned greedy policy and requires it to beat the *stick on 20+* baseline by a wide margin.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 05-monte-carlo/02-mc-control
import numpy as np
import rlhud
from rlenvs import Blackjack


def epsilon_greedy(q, epsilon, rng):
    """Pick an action from the action-value array `q` (one entry per action).

    With probability epsilon: a uniformly random action.
    Otherwise: the greedy action argmax(q) (ties may be broken any way you like).
    `rng` is a numpy Generator (np.random.default_rng).
    """
    # TODO
    raise NotImplementedError


def mc_control(env, n_episodes, epsilon=0.1, gamma=1.0, seed=None):
    """On-policy first-visit Monte Carlo control with an epsilon-soft (epsilon-greedy) policy.

    Returns (Q, returns):
      Q       : dict {state: np.array of shape (env.n_actions,)}
      returns : list with the total (undiscounted) reward of every training episode
    """
    rng = np.random.default_rng(seed)
    n_actions = env.n_actions
    Q = {}
    N = {}  # visit counts: {state: np.zeros(n_actions)}
    returns = []
    for i in range(n_episodes):
        # TODO 1: generate an episode acting epsilon-greedily on Q
        #         (create Q[obs] / N[obs] the first time you see obs; seed only the first reset)
        # TODO 2: append the episode's total reward to `returns`
        # TODO 3: walk the episode backwards (G = r + gamma * G) and, for the FIRST visit of each
        #         (s, a), update Q[s][a] towards G with an incremental mean
        raise NotImplementedError
    return Q, returns


def evaluate(env, policy, n_episodes, seed=0):
    """Mean return of a deterministic policy(obs) -> action over n_episodes (no learning)."""
    total = 0.0
    obs, _ = env.reset(seed=seed)
    for i in range(n_episodes):
        if i:
            obs, _ = env.reset()
        while True:
            obs, r, term, trunc, _ = env.step(policy(obs))
            total += r
            if term or trunc:
                break
    return total / n_episodes


if __name__ == "__main__":
    env = Blackjack(seed=1)
    N_EPISODES = 200_000
    Q, rets = mc_control(env, N_EPISODES, epsilon=0.1, seed=1)

    # Mean training return per block of 2000 episodes
    block = 2000
    curve = [float(np.mean(rets[i:i + block])) for i in range(0, len(rets), block)]
    rlhud.plot(curve, label="ε-greedy MC (training)", title="Mean return per 2000 episodes",
               xlabel="block", ylabel="mean return")

    greedy = lambda s: int(np.argmax(Q[s])) if s in Q else 0
    naive = lambda s: 0 if s[0] >= 20 else 1
    g_score = evaluate(Blackjack(), greedy, 50_000, seed=123)
    n_score = evaluate(Blackjack(), naive, 50_000, seed=123)
    rlhud.table([["stick on 20+", round(n_score, 4)], ["learned greedy policy", round(g_score, 4)]],
                headers=["policy", "mean return (50k hands)"], title="Head to head")
    rlhud.stat("Learned policy mean return", round(g_score, 4), good=g_score > n_score)

    # Policy maps in S&B Figure 5.2 style: 1 = HIT, 0 = STICK
    sums = list(range(21, 10, -1))
    dealer = list(range(1, 11))
    for usable in (True, False):
        pol = [[greedy((p, d, usable)) for d in dealer] for p in sums]
        labels = [["H" if v else "S" for v in row] for row in pol]
        rlhud.grid(pol, title=f"Greedy policy, {'usable' if usable else 'no usable'} ace "
                              f"(rows 21→11, cols dealer A→10; H = hit)", labels=labels, cmap="heat")

In [ ]:
#@title ✅ **Check** · On-policy MC control on Blackjack (+150 XP) — tests the code you last ran above
rlcolab.check("05-monte-carlo/02-mc-control")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

There is no separate 'improvement' step: because every new episode acts ε-greedily on the *latest* Q, evaluation and improvement are interleaved — generalised policy iteration, one episode at a time.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Keep a second dict `N = {state: np.zeros(n_actions)}` of visit counts next to Q. Create both entries when you first see a state, *before* calling epsilon_greedy on it.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

After generating `episode`:

first = {}
for t, (s, a, _) in enumerate(episode): first.setdefault((s, a), t)
G = 0.0
for t in range(len(episode) - 1, -1, -1):
    s, a, r = episode[t]
    G = r + gamma * G
    if first[(s, a)] == t:
        N[s][a] += 1
        Q[s][a] += (G - Q[s][a]) / N[s][a]

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("05-monte-carlo/02-mc-control")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 05-monte-carlo/02-mc-control
# Reference solution
import numpy as np
import rlhud
from rlenvs import Blackjack


def epsilon_greedy(q, epsilon, rng):
    """Pick an action from the action-value array `q` (one entry per action).

    With probability epsilon: a uniformly random action.
    Otherwise: the greedy action argmax(q) (ties may be broken any way you like).
    `rng` is a numpy Generator (np.random.default_rng).
    """
    if rng.random() < epsilon:
        return int(rng.integers(len(q)))
    return int(np.argmax(q))


def mc_control(env, n_episodes, epsilon=0.1, gamma=1.0, seed=None):
    """On-policy first-visit Monte Carlo control with an epsilon-soft (epsilon-greedy) policy.

    Returns (Q, returns):
      Q       : dict {state: np.array of shape (env.n_actions,)}
      returns : list with the total (undiscounted) reward of every training episode
    """
    rng = np.random.default_rng(seed)
    n_actions = env.n_actions
    Q = {}
    N = {}  # visit counts per (state) -> array per action, for incremental averaging
    returns = []
    for i in range(n_episodes):
        # 1. generate an episode with the CURRENT epsilon-greedy policy
        episode = []
        obs, _ = env.reset(seed=seed if i == 0 else None)
        while True:
            if obs not in Q:
                Q[obs] = np.zeros(n_actions)
                N[obs] = np.zeros(n_actions)
            a = epsilon_greedy(Q[obs], epsilon, rng)
            next_obs, r, terminated, truncated, _ = env.step(a)
            episode.append((obs, a, r))
            if terminated or truncated:
                break
            obs = next_obs
        returns.append(sum(r for _, _, r in episode))

        # 2. first-visit MC update of Q towards the observed returns
        first = {}
        for t, (s, a, _) in enumerate(episode):
            first.setdefault((s, a), t)
        G = 0.0
        for t in range(len(episode) - 1, -1, -1):
            s, a, r = episode[t]
            G = r + gamma * G
            if first[(s, a)] == t:
                N[s][a] += 1
                Q[s][a] += (G - Q[s][a]) / N[s][a]   # incremental mean
        # 3. policy improvement is implicit: the next episode acts epsilon-greedily on the new Q
    return Q, returns


def evaluate(env, policy, n_episodes, seed=0):
    """Mean return of a deterministic policy(obs) -> action over n_episodes (no learning)."""
    total = 0.0
    obs, _ = env.reset(seed=seed)
    for i in range(n_episodes):
        if i:
            obs, _ = env.reset()
        while True:
            obs, r, term, trunc, _ = env.step(policy(obs))
            total += r
            if term or trunc:
                break
    return total / n_episodes


if __name__ == "__main__":
    env = Blackjack(seed=1)
    N_EPISODES = 200_000
    Q, rets = mc_control(env, N_EPISODES, epsilon=0.1, seed=1)

    # Mean training return per block of 2000 episodes
    block = 2000
    curve = [float(np.mean(rets[i:i + block])) for i in range(0, len(rets), block)]
    rlhud.plot(curve, label="ε-greedy MC (training)", title="Mean return per 2000 episodes",
               xlabel="block", ylabel="mean return")

    greedy = lambda s: int(np.argmax(Q[s])) if s in Q else 0
    naive = lambda s: 0 if s[0] >= 20 else 1
    g_score = evaluate(Blackjack(), greedy, 50_000, seed=123)
    n_score = evaluate(Blackjack(), naive, 50_000, seed=123)
    rlhud.table([["stick on 20+", round(n_score, 4)], ["learned greedy policy", round(g_score, 4)]],
                headers=["policy", "mean return (50k hands)"], title="Head to head")
    rlhud.stat("Learned policy mean return", round(g_score, 4), good=g_score > n_score)

    # Policy maps in S&B Figure 5.2 style: 1 = HIT, 0 = STICK
    sums = list(range(21, 10, -1))
    dealer = list(range(1, 11))
    for usable in (True, False):
        pol = [[greedy((p, d, usable)) for d in dealer] for p in sums]
        labels = [["H" if v else "S" for v in row] for row in pol]
        rlhud.grid(pol, title=f"Greedy policy, {'usable' if usable else 'no usable'} ace "
                              f"(rows 21→11, cols dealer A→10; H = hit)", labels=labels, cmap="heat")

> **🔑 KEY IDEA** · **On-policy MC control in one sentence**
>
> Play an episode ε-greedily with respect to Q, then move each visited Q(s, a) towards the return
> that actually followed it. Repeat. Evaluation and improvement happen at the same time.

The policy your agent finds should look like the famous *basic strategy* charts: always hit on 11
or less, stick on hard 17+, and for hard 13–16, stick when the dealer shows a weak card (2–6) and
hit when it shows a strong one (7–A). Expect a noisy version after 200 000 hands. A dealer showing a 5 or 6 is likely to bust, so you let them.

## 04 · Off-policy learning

The ε-greedy agent above learns about the policy it *follows*, exploration noise included. That
is **on-policy** learning. Often we want something different:

- learn the **greedy** policy while behaving exploratorily,
- learn from data produced by someone else (a human, an old version of the agent, a log file),
- evaluate a new policy *before* deploying it.

That is **off-policy** learning. Two policies are now involved:

| | role |
|---|---|
| **target policy** π | the one we want to learn about (often greedy) |
| **behaviour policy** b | the one that actually generated the data (must explore) |

For this to be possible, b must try everything π might do: if π(a|s) > 0 then b(a|s) > 0.
This is the **coverage** assumption.

### Importance sampling

Returns collected under b have the wrong expectation: E_b[G] = v_b, not v_π. The fix is to
re-weight each return by how much more (or less) likely its trajectory is under π than under b.
The environment's transition probabilities appear in both and cancel, leaving only the policies:

```text
ρ = Πₜ π(Aₜ|Sₜ) / b(Aₜ|Sₜ)
```

ρ is the **importance-sampling ratio**. If π is deterministic and the behaviour took an action π
would never take, ρ = 0 and that episode tells us nothing. If b took π's action with probability
½ at each of three steps, ρ = 2³ = 8 — that rare trajectory is worth eight ordinary ones.

Two ways to average the weighted returns from n episodes:

```text
ordinary:  V(s) = Σᵢ ρᵢ Gᵢ / n          weighted:  V(s) = Σᵢ ρᵢ Gᵢ / Σᵢ ρᵢ
```

| | bias | variance | intuition |
|---|---|---|---|
| **ordinary IS** | unbiased | can be huge, even infinite | one lucky ρ = 64 episode dominates everything |
| **weighted IS** | biased (→ 0 as n grows) | bounded, much lower | a weighted *average* of real returns, so it never leaves their range |

After one episode with ρ = 4 and G = 2, ordinary IS says 8 — a Blackjack return of 8 is
impossible! Weighted IS says 2, the return actually observed. In practice weighted IS is almost
always preferred.

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("05-monte-carlo#2", {"q": "Target π: always HIT. Behaviour b: hit or stick 50/50. An episode is hit, hit, stick. What is ρ?", "options": ["0.125", "8", "0", "4"], "answer": 2, "explain": "π never sticks, so π(stick|s) = 0 and the whole product is 0. That trajectory is impossible under π and contributes nothing."})

---
### 🧪 Exercise 3 · Ordinary vs weighted importance sampling · ⭐ **130 XP**

Evaluate a policy you never actually follow. The data comes from a **random** behaviour policy *b*; you want the value of the **stick on 20+** target policy π in one Blackjack state (soft 13 vs dealer 2 — Sutton & Barto Example 5.4, true value ≈ −0.277).

Implement `is_estimates(episodes, pi, b, gamma=1.0)`. Every episode starts in the same state. For episode *i* compute

- its return from the start, `Gᵢ = Σₜ γᵗ Rₜ₊₁`
- its importance-sampling ratio, `ρᵢ = Πₜ π(Aₜ|Sₜ) / b(Aₜ|Sₜ)` (call `pi(a, s)` and `b(a, s)`)

and return two lists `(ordinary, weighted)` whose element *k* is the estimate after the first *k+1* episodes:

- ordinary: `Σ ρᵢGᵢ / (k+1)`
- weighted: `Σ ρᵢGᵢ / Σ ρᵢ` — and `0.0` while `Σ ρᵢ` is still 0.

The helper `play_from_soft13_vs_2`, both policies and `TRUE_VALUE` are provided. **Run** reproduces Figure 5.3: the error of both estimators over 2000 episodes.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 05-monte-carlo/03-importance-sampling
import random

import numpy as np
import rlhud
from rlenvs import Blackjack


def is_estimates(episodes, pi, b, gamma=1.0):
    """Off-policy Monte Carlo estimates of v_pi(start state) from behaviour-policy episodes.

    episodes : list of episodes, each a list of (state, action, reward), all from the SAME start state
    pi(a, s) : probability that the TARGET policy takes action a in state s
    b(a, s)  : probability that the BEHAVIOUR policy (which generated the data) takes a in s

    Returns (ordinary, weighted): two lists of the same length as `episodes`; element k is the
    estimate after the first k+1 episodes.
        ordinary : sum(rho_i * G_i) / (number of episodes)
        weighted : sum(rho_i * G_i) / sum(rho_i)      (0.0 while sum(rho_i) == 0)
    """
    ordinary, weighted = [], []
    sum_wg, sum_w = 0.0, 0.0
    for k, episode in enumerate(episodes, start=1):
        # TODO: compute this episode's ratio rho and its return G from the start state,
        #       update the running sums, then append both estimates.
        raise NotImplementedError
    return ordinary, weighted


# ---------------------------------------------------------------- provided
def play_from_soft13_vs_2(env, policy):
    """One Blackjack episode starting in the S&B Example 5.4 state:
    player holds A + 2 (= soft 13, usable ace), dealer shows a 2.
    policy(s) -> action. Returns the list of (state, action, reward)."""
    env.reset()
    env.player = [1, 2]
    env.dealer = [2, env._card()]
    obs, episode = env._obs(), []
    while True:
        a = policy(obs)
        nxt, r, term, trunc, _ = env.step(a)
        episode.append((obs, a, r))
        if term or trunc:
            return episode
        obs = nxt


def target_pi(a, s):
    """Target policy: stick on 20 or 21, otherwise hit (deterministic)."""
    return 1.0 if a == (0 if s[0] >= 20 else 1) else 0.0


def behaviour_b(a, s):
    """Behaviour policy: hit or stick with equal probability."""
    return 0.5


TRUE_VALUE = -0.27726   # v_pi of that state (Sutton & Barto, Example 5.4)


if __name__ == "__main__":
    random.seed(0)
    env = Blackjack(seed=0)
    random_policy = lambda s: random.randrange(2)
    RUNS, EPISODES = 40, 2000
    sq_ord = np.zeros(EPISODES)
    sq_w = np.zeros(EPISODES)
    for run in range(RUNS):
        eps = [play_from_soft13_vs_2(env, random_policy) for _ in range(EPISODES)]
        o, w = is_estimates(eps, target_pi, behaviour_b)
        sq_ord += (np.array(o) - TRUE_VALUE) ** 2
        sq_w += (np.array(w) - TRUE_VALUE) ** 2
        rlhud.progress(run, RUNS, "Runs")
    idx = np.unique(np.logspace(0, np.log10(EPISODES), 60).astype(int)) - 1
    rlhud.plot((sq_ord / RUNS)[idx], x=idx + 1, label="ordinary IS", title="Mean squared error vs episodes (cf. S&B Fig 5.3)",
               xlabel="episodes", ylabel="MSE")
    rlhud.plot((sq_w / RUNS)[idx], x=idx + 1, label="weighted IS", title="Mean squared error vs episodes (cf. S&B Fig 5.3)")
    rlhud.stat("Final MSE, ordinary", round(float(sq_ord[-1] / RUNS), 4))
    rlhud.stat("Final MSE, weighted", round(float(sq_w[-1] / RUNS), 4), good=True)
    print("Weighted IS is biased early on but has far lower variance — it wins on almost every run.")

In [ ]:
#@title ✅ **Check** · Ordinary vs weighted importance sampling (+130 XP) — tests the code you last ran above
rlcolab.check("05-monte-carlo/03-importance-sampling")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Keep two running sums across episodes: `sum_wg` (Σ ρG) and `sum_w` (Σ ρ). Then both estimates are one division each.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Inside one episode: start `rho = 1.0, G = 0.0`; for each step t, `rho *= pi(a, s) / b(a, s)` and `G += gamma**t * r`. If π would never take the action, ρ becomes 0 and the episode contributes nothing to the numerator.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

for k, ep in enumerate(episodes, start=1):
    rho, G = 1.0, 0.0
    for t, (s, a, r) in enumerate(ep):
        rho *= pi(a, s) / b(a, s); G += gamma**t * r
    sum_wg += rho * G; sum_w += rho
    ordinary.append(sum_wg / k)
    weighted.append(sum_wg / sum_w if sum_w > 0 else 0.0)

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("05-monte-carlo/03-importance-sampling")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 05-monte-carlo/03-importance-sampling
# Reference solution
import random

import numpy as np
import rlhud
from rlenvs import Blackjack


def is_estimates(episodes, pi, b, gamma=1.0):
    """Off-policy Monte Carlo estimates of v_pi(start state) from behaviour-policy episodes.

    episodes : list of episodes, each a list of (state, action, reward), all from the SAME start state
    pi(a, s) : probability that the TARGET policy takes action a in state s
    b(a, s)  : probability that the BEHAVIOUR policy (which generated the data) takes a in s

    Returns (ordinary, weighted): two lists of the same length as `episodes`; element k is the
    estimate after the first k+1 episodes.
        ordinary : sum(rho_i * G_i) / (number of episodes)
        weighted : sum(rho_i * G_i) / sum(rho_i)      (0.0 while sum(rho_i) == 0)
    """
    ordinary, weighted = [], []
    sum_wg, sum_w = 0.0, 0.0
    for k, episode in enumerate(episodes, start=1):
        rho, G = 1.0, 0.0
        for t, (s, a, r) in enumerate(episode):
            rho *= pi(a, s) / b(a, s)       # importance-sampling ratio of the whole trajectory
            G += (gamma ** t) * r           # return from the start state
        sum_wg += rho * G
        sum_w += rho
        ordinary.append(sum_wg / k)
        weighted.append(sum_wg / sum_w if sum_w > 0 else 0.0)
    return ordinary, weighted


# ---------------------------------------------------------------- provided
def play_from_soft13_vs_2(env, policy):
    """One Blackjack episode starting in the S&B Example 5.4 state:
    player holds A + 2 (= soft 13, usable ace), dealer shows a 2.
    policy(s) -> action. Returns the list of (state, action, reward)."""
    env.reset()
    env.player = [1, 2]
    env.dealer = [2, env._card()]
    obs, episode = env._obs(), []
    while True:
        a = policy(obs)
        nxt, r, term, trunc, _ = env.step(a)
        episode.append((obs, a, r))
        if term or trunc:
            return episode
        obs = nxt


def target_pi(a, s):
    """Target policy: stick on 20 or 21, otherwise hit (deterministic)."""
    return 1.0 if a == (0 if s[0] >= 20 else 1) else 0.0


def behaviour_b(a, s):
    """Behaviour policy: hit or stick with equal probability."""
    return 0.5


TRUE_VALUE = -0.27726   # v_pi of that state (Sutton & Barto, Example 5.4)


if __name__ == "__main__":
    random.seed(0)
    env = Blackjack(seed=0)
    random_policy = lambda s: random.randrange(2)
    RUNS, EPISODES = 40, 2000
    sq_ord = np.zeros(EPISODES)
    sq_w = np.zeros(EPISODES)
    for run in range(RUNS):
        eps = [play_from_soft13_vs_2(env, random_policy) for _ in range(EPISODES)]
        o, w = is_estimates(eps, target_pi, behaviour_b)
        sq_ord += (np.array(o) - TRUE_VALUE) ** 2
        sq_w += (np.array(w) - TRUE_VALUE) ** 2
        rlhud.progress(run, RUNS, "Runs")
    idx = np.unique(np.logspace(0, np.log10(EPISODES), 60).astype(int)) - 1
    rlhud.plot((sq_ord / RUNS)[idx], x=idx + 1, label="ordinary IS", title="Mean squared error vs episodes (cf. S&B Fig 5.3)",
               xlabel="episodes", ylabel="MSE")
    rlhud.plot((sq_w / RUNS)[idx], x=idx + 1, label="weighted IS", title="Mean squared error vs episodes (cf. S&B Fig 5.3)")
    rlhud.stat("Final MSE, ordinary", round(float(sq_ord[-1] / RUNS), 4))
    rlhud.stat("Final MSE, weighted", round(float(sq_w[-1] / RUNS), 4), good=True)
    print("Weighted IS is biased early on but has far lower variance — it wins on almost every run.")

### Off-policy MC control (in brief)

Put the pieces together: behave with a soft b (e.g. ε-greedy), keep a greedy target π, and update
Q with weighted importance sampling, walking each episode backwards. Because π is greedy,
ρ becomes 0 the moment b takes a non-greedy action, so the algorithm only learns from the
**tails** of episodes after the last exploratory move. That makes it slow when exploratory
actions are common — one motivation for the temporal-difference methods in the next lesson.

## 05 · Pitfalls

> **⚠️ WARNING** · **Common Monte Carlo bugs**
>
> - **Wrong reward alignment.** The triple is (Sₜ, Aₜ, Rₜ₊₁): the reward that *follows* the action.
>   Storing the next state instead of the current one shifts every return by one step.
> - **Forward instead of backward.** Computing each Gₜ by summing forward is O(T²). Walk backwards once.
> - **Re-seeding every episode.** `env.reset(seed=0)` inside the loop replays the *same* deal forever.
>   Seed once, then let the environment's RNG run.
> - **Greedy too early.** With ε = 0 and Q initialised to zeros, argmax always picks action 0 —
>   some actions will never be tried.
> - **Ordinary IS explosions.** Long episodes multiply many ratios; ρ can be astronomically large.
>   Use weighted IS, or keep episodes short.

## 06 · Summary

- MC estimates values by **averaging sample returns**; it needs no model and does not bootstrap.
- **First-visit** and **every-visit** MC both converge to v_π; first-visit is unbiased.
- Control estimates **Q(s, a)** and runs GPI episode by episode, with **ε-soft** policies (or
  exploring starts) to keep exploring.
- **Off-policy** learning evaluates π from data generated by b using the **importance-sampling
  ratio**; **weighted** IS trades a little bias for a big drop in variance.

The weakness of every method here: nothing is learned until the episode ends. Next lesson we fix
that by updating a guess from a guess — **temporal-difference learning**.

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("05-monte-carlo")